# Test: `torch.linspace` CPU/CUDA Integer Precision Fix

**Issue:** [pytorch/pytorch#181807](https://github.com/pytorch/pytorch/issues/181807)

This notebook verifies the fix for `torch.linspace` producing inconsistent
integer results between CPU and CUDA.

**Instructions:**
1. Open in Google Colab (Runtime > Change runtime type > GPU)
2. Run all cells
3. If all tests pass, the fix is working correctly

## Step 1: Install PyTorch from your fork (with the fix)

In [ ]:
# Clone the fix files and official PyTorch
!git clone https://github.com/gaurav-redhat/TestRepo.git
!git clone --depth 1 https://github.com/pytorch/pytorch.git pytorch-fix
%cd pytorch-fix
!git submodule update --init --recursive --depth 1

# Replace original files with fixed versions
!cp ../TestRepo/181807/RangeFactories.cu aten/src/ATen/native/cuda/RangeFactories.cu
!cp ../TestRepo/181807/test_tensor_creation_ops.py test/test_tensor_creation_ops.py
!echo 'Fixed files applied successfully!'

In [ ]:
# Build PyTorch from source (this takes ~30-60 min on Colab)
!pip install -r requirements.txt
!python setup.py develop 2>&1 | tail -5

## Step 2: Verify the fix

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
def test_linspace_cpu_cuda_match(start, end, steps, dtype):
    """Test that linspace gives the same result on CPU and CUDA."""
    cpu = torch.linspace(start, end, steps, dtype=dtype, device="cpu")
    gpu = torch.linspace(start, end, steps, dtype=dtype, device="cuda").cpu()

    match = torch.equal(cpu, gpu)
    status = "PASS" if match else "FAIL"

    print(f"[{status}] linspace({start}, {end}, {steps}, dtype={dtype})")
    if not match:
        diff_idx = (cpu != gpu).nonzero(as_tuple=True)[0].tolist()
        print(f"  CPU: {cpu.tolist()}")
        print(f"  GPU: {gpu.tolist()}")
        print(f"  Differ at indices: {diff_idx}")
    return match

In [ ]:
print("=" * 60)
print("Test: Original reproducer from issue #181807")
print("=" * 60)
test_linspace_cpu_cuda_match(3.7, -3, 10, torch.int64)

In [ ]:
print("=" * 60)
print("Test: Additional edge cases")
print("=" * 60)

all_passed = True
test_cases = [
    (3.7, -3, 10, torch.int64),
    (0, 100, 51, torch.int64),
    (-50, 50, 101, torch.int64),
    (0, 1000000, 999, torch.int64),
    (1.5, 10.5, 20, torch.int32),
    (-100.7, 100.3, 200, torch.int64),
    (0, 255, 256, torch.int16),
    (10, -10, 5, torch.int64),
    (0.1, 0.9, 5, torch.int64),
    (1e6, -1e6, 100, torch.int64),
]

for start, end, steps, dtype in test_cases:
    if not test_linspace_cpu_cuda_match(start, end, steps, dtype):
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL TESTS PASSED - Fix is working!")
else:
    print("SOME TESTS FAILED - Fix needs more work.")
print("=" * 60)

## Step 3: Quick test WITHOUT the fix (optional)

Run this cell to see the bug on stock PyTorch (before the fix).

In [ ]:
# Uncomment to test with stock PyTorch (shows the bug)
# !pip install torch --index-url https://download.pytorch.org/whl/cu121 --force-reinstall -q
# import importlib, torch
# importlib.reload(torch)
# cpu = torch.linspace(3.7, -3, 10, dtype=torch.int64, device="cpu")
# gpu = torch.linspace(3.7, -3, 10, dtype=torch.int64, device="cuda").cpu()
# print("cpu:", cpu.tolist())
# print("gpu:", gpu.tolist())
# print("match:", torch.equal(cpu, gpu))